# Train SmolVLA trên dataset CARLA (vla-drive), trên Google Colab

Notebook này train đúng như lệnh trên máy bạn (`docs/carla.md`), nhưng trên GPU của Colab.

**Trên máy bạn, trước khi mở notebook:**

```bash
make carla-colab-pack      # -> runs/carla/carla-lerobot-train.tar (khoảng 510 MB: 105 episode, 63 000 frame)
```

Upload file đó vào Google Drive, thư mục `MyDrive/vla-drive/`.

**Trên Colab:**

1. *Runtime → Change runtime type → GPU* (T4 miễn phí; L4/A100 nếu có Colab Pro).
2. Sửa ô **Thông số** nếu cần, rồi *Runtime → Run all*.
3. **Bị ngắt giữa chừng?** Mở lại notebook, *Run all*: nó chép checkpoint mới nhất từ
   Drive về và chạy tiếp từ đó. Drive chỉ giữ checkpoint mới nhất (khoảng 1,3 GB).
4. Xong: tải `MyDrive/vla-drive/<RUN>-pretrained_model.tar` (khoảng 870 MB) về máy, rồi:

```bash
make carla-colab-unpack TAR=~/Downloads/smolvla-carla-v1-pretrained_model.tar COLAB_RUN=smolvla-carla-v1
./vla.sh tools/probe_vla_carla.py --checkpoint smolvla-carla-v1      # kiểm offline
make carla-server TOWN=Town01
make carla-drive TOWN=Town01 ARGS='--checkpoint smolvla-carla-v1 --episodes 10 --show'
```

Drive cần trống khoảng 3 GB. Trên RTX 4060 laptop, 8000 bước batch 16 mất khoảng 50 phút
(2,75 bước/giây); tốc độ trên Colab xem ở `updt_s` trong log (giây mỗi bước).

## 1. GPU và Python

In [ ]:
import subprocess, sys
print(sys.version)
assert sys.version_info >= (3, 12), "LeRobot 0.5.1 cần Python 3.12 trở lên; runtime này cũ hơn"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Không có GPU: Runtime → Change runtime type → GPU"
print("GPU:", gpu)

## 2. Thông số

In [ ]:
# ---- Thông số: chỉ cần sửa ở đây ----
DRIVE_DIR = "/content/drive/MyDrive/vla-drive"   # thư mục trên Drive chứa file .tar
DATASET_TAR = "carla-lerobot-train.tar"          # do `make carla-colab-pack` tạo trên máy bạn
RUN = "smolvla-carla-v1"                         # tên lượt train; chạy lại cùng tên là chạy tiếp
STEPS = 8000                                     # khoảng 2 lượt qua 63 000 frame ở batch 16
BATCH = 16                                       # A100: 32 được (rồi STEPS = 4000 cho cùng số mẫu)
SAVE_FREQ = 1000                                 # bước giữa hai checkpoint
LOG_FREQ = 100
VIDEO_BACKEND = "pyav"                           # giải AV1 bằng libdav1d; khỏi lo torchcodec lệch bản torch
WORK = "/content"                                # ổ của Colab: nhanh, mất khi phiên kết thúc

## 3. Google Drive

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
tar = Path(DRIVE_DIR) / DATASET_TAR
assert tar.is_file(), f"Không thấy {tar}. Trên máy bạn: make carla-colab-pack, rồi upload file đó vào {DRIVE_DIR}"
print(f"{tar}: {tar.stat().st_size / 1e6:.0f} MB")

## 4. Cài LeRobot 0.5.1

In [ ]:
# Đúng bản trên máy bạn: checkpoint train bằng bản khác có thể không load được ở máy bạn.
!pip install -q "lerobot[smolvla]==0.5.1"
import importlib.metadata as md
for name in ("lerobot", "torch", "av"):
    print(name, md.version(name))

## 5. Dataset: giải nén ra ổ của Colab

In [ ]:
import json, subprocess
from pathlib import Path
data_root = Path(WORK) / "data" / "lerobot-train"
if not (data_root / "meta" / "info.json").is_file():           # đọc video thẳng từ Drive thì rất chậm
    data_root.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xf", str(Path(DRIVE_DIR) / DATASET_TAR), "-C", str(data_root.parent)], check=True)
info = json.loads((data_root / "meta" / "info.json").read_text())
print(f"{data_root}: {info['total_episodes']} episodes, {info['total_frames']} frames, {info['fps']} fps")

## 6. Các hàm: chạy tiếp, và giữ checkpoint mới nhất trên Drive

In [ ]:
import os, shutil, subprocess, threading
from pathlib import Path
out = Path(WORK) / "runs" / RUN               # lerobot-train ghi vào đây (ổ local)
saved = Path(DRIVE_DIR) / "runs" / RUN        # trên Drive: chỉ checkpoint mới nhất, và log


def checkpoints(root):
    """The step directories under root/checkpoints, oldest first."""
    folder = root / "checkpoints"
    if not folder.is_dir():
        return []
    return sorted((p for p in folder.iterdir() if p.is_dir() and p.name.isdigit()), key=lambda p: int(p.name))


def point_last(step_dir):
    last = out / "checkpoints" / "last"
    if last.is_symlink() or last.exists():
        last.unlink()
    last.symlink_to(step_dir.name)


def restore():
    """The checkpoint to resume from: this session's, else the newest on Drive, copied back to
    where the saved train config expects it. None: start from smolvla_base."""
    last = out / "checkpoints" / "last"
    if last.is_symlink() and (last / "pretrained_model" / "train_config.json").is_file():
        return last.resolve()
    on_drive = checkpoints(saved)
    if not on_drive:
        return None
    newest = on_drive[-1]
    local = out / "checkpoints" / newest.name
    if not local.exists():
        print(f"restoring checkpoint {newest.name} from Drive ...", flush=True)
        shutil.copytree(newest, local)
    point_last(local)
    return local


def keep_newest(stop):
    """Copy each checkpoint lerobot-train finishes to Drive, and drop the older ones on both sides.
    lerobot-train points `last` at a checkpoint once it has written all of it."""
    (saved / "checkpoints").mkdir(parents=True, exist_ok=True)
    done = {p.name for p in checkpoints(saved)}
    while True:
        last = out / "checkpoints" / "last"
        if last.is_symlink():
            step = os.readlink(last)
            if step not in done:
                partial = saved / "checkpoints" / f".{step}.partial"
                shutil.rmtree(partial, ignore_errors=True)
                shutil.copytree(out / "checkpoints" / step, partial)
                partial.rename(saved / "checkpoints" / step)
                done.add(step)
                for root in (saved, out):
                    for old in checkpoints(root):
                        if int(old.name) < int(step):
                            shutil.rmtree(old, ignore_errors=True)
                print(f"[drive] checkpoint {step} -> {saved / 'checkpoints' / step}", flush=True)
        if stop.is_set():
            return
        stop.wait(30)


def shown(line):
    """What of lerobot-train's output to print: its step and checkpoint lines; not the progress
    bar (a line per step) nor torchvision's notice (once per data loader worker)."""
    if "Training:" in line:
        at = line.find("INFO")               # lerobot writes its step line on the end of the bar
        return line[at:] if at >= 0 else None
    if "video_deprecation_warning" in line or line.strip() == "warnings.warn(":
        return None
    return line


def train_command(resume_from):
    if resume_from is not None:
        return ["lerobot-train", f"--config_path={out}/checkpoints/last/pretrained_model/train_config.json",
                "--resume=true", f"--steps={STEPS}"]
    return ["lerobot-train", "--policy.path=lerobot/smolvla_base",
            "--dataset.repo_id=local/vla-drive", f"--dataset.root={data_root}",
            f"--dataset.video_backend={VIDEO_BACKEND}",
            '--rename_map={"observation.images.front": "observation.images.camera1"}',
            "--policy.chunk_size=16", "--policy.n_action_steps=16",
            f"--batch_size={BATCH}", f"--steps={STEPS}", f"--log_freq={LOG_FREQ}", f"--save_freq={SAVE_FREQ}",
            "--policy.push_to_hub=false", "--policy.device=cuda", "--wandb.enable=false",
            f"--output_dir={out}", f"--job_name={RUN}"]

## 7. Train

Mỗi `LOG_FREQ` bước một dòng `step:... loss:... updt_s:...`. Mỗi khi có checkpoint, dòng `[drive]` báo đã chép lên Drive (tối đa 30 s sau).

In [ ]:
resume_from = restore()
if resume_from is None and out.exists():
    shutil.rmtree(out)                       # a run that never reached a checkpoint: start it again
command = train_command(resume_from)
print("resuming from step", resume_from.name if resume_from else "-", "|", " ".join(command[:3]), "...", flush=True)

saved.mkdir(parents=True, exist_ok=True)
stop = threading.Event()
syncer = threading.Thread(target=keep_newest, args=(stop,), daemon=True)
syncer.start()
env = dict(os.environ, PYTHONUNBUFFERED="1")
with open(saved / "train.log", "a") as log, subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1) as proc:
    for line in proc.stdout:
        log.write(line)                      # everything, on Drive
        text = shown(line)
        if text:
            print(text, end="", flush=True)
    code = proc.wait()
stop.set()
syncer.join()
print("lerobot-train exited with", code, "| newest checkpoint on Drive:", [p.name for p in checkpoints(saved)])
assert code == 0, f"lerobot-train failed; the whole log is in {saved / 'train.log'}"

## 8. Đóng gói model để tải về

In [ ]:
# The model alone (no optimizer state), to take home: about 870 MB.
if not (out / "checkpoints" / "last").is_symlink():
    restore()                                # a new session that only packs
step_dir = (out / "checkpoints" / "last").resolve()
package = Path(DRIVE_DIR) / f"{RUN}-pretrained_model.tar"
subprocess.run(["tar", "-cf", str(package), "-C", str(step_dir), "pretrained_model"], check=True)
print(f"{package}: {package.stat().st_size / 1e6:.0f} MB, step {step_dir.name}")
print("On your machine: download it, then")
print(f"  make carla-colab-unpack TAR=~/Downloads/{package.name} COLAB_RUN={RUN}")